In [1]:
# ============================================================
# CELL 1 — Benchmark data partition and provenance
# ============================================================
# Establish the approved chronological development and
# evaluation partitions without modifying original data.

from pathlib import Path
from datetime import datetime
import hashlib
import csv
import json
import sys

# Locate the project and verify the Python environment.
ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "PROTOCOL.md").is_file()
)

DATA = ROOT / "data"
RAW = DATA / "raw"

assert Path(sys.prefix).resolve() == (
    ROOT / ".venv"
).resolve()

SOURCE_COMMIT = (DATA / "source_commit.txt").read_text().strip()

# Verify the integrity of the original source files.
manifest = {}

for line in (DATA / "SHA256SUMS").read_text().splitlines():
    digest, filename = line.split(maxsplit=1)
    manifest[filename] = digest

actual_files = {
    path.relative_to(ROOT).as_posix()
    for path in RAW.rglob("*")
    if path.is_file()
}

assert actual_files == set(manifest)

for filename, expected_hash in manifest.items():
    assert hashlib.sha256(
        (ROOT / filename).read_bytes()
    ).hexdigest() == expected_hash

# Load the original batch definitions.
proofs = json.loads(
    (RAW / "Meta Data/proof_log.json").read_text()
)

# Preserve the original CSV records and row positions.
records_by_file = {}

for file in sorted((RAW / "logs").glob("*.csv")):
    with file.open(encoding="utf-8", newline="") as stream:
        records_by_file[file.name] = list(csv.DictReader(stream))

# Approved experimental allocation.
# Source: EXPERIMENTAL_DESIGN.md, Section 2.
DEVELOPMENT_COUNT = 65
EVALUATION_COUNT = 65

assert len(proofs) == DEVELOPMENT_COUNT + EVALUATION_COUNT

# Reconstruct and audit the chronological partitions.
development_batches = []
evaluation_batch_indices = []

development_hashes = set()
evaluation_hashes = set()

partition_times = {
    "development": [],
    "evaluation": [],
}

previous_end = None

for index, proof in enumerate(proofs):

    filename = Path(proof["csv_path"]).name
    start = proof["row_start"]
    end = proof["row_end"]

    assert 0 <= start <= end < len(records_by_file[filename])

    records = records_by_file[filename][start:end + 1]

    assert len(records) == proof["size"]

    timestamps = [
        datetime.fromisoformat(record["timestamp"])
        for record in records
    ]

    assert all(
        current > previous
        for previous, current in zip(
            timestamps, timestamps[1:]
        )
    )

    first_time = timestamps[0]
    last_time = timestamps[-1]

    if previous_end is not None:
        assert first_time > previous_end

    previous_end = last_time

    hashes = {
        record["chain_hash"] for record in records
    }

    assert len(hashes) == len(records)

    if index < DEVELOPMENT_COUNT:

        assert not development_hashes.intersection(hashes)
        development_hashes.update(hashes)

        development_batches.append({
            "batch_index": index,
            "proof": proof,
            "records": records,
        })

        partition_times["development"].extend(
            [first_time, last_time]
        )

    else:

        assert not evaluation_hashes.intersection(hashes)
        evaluation_hashes.update(hashes)

        evaluation_batch_indices.append(index)

        partition_times["evaluation"].extend(
            [first_time, last_time]
        )

# Verify separation between the two partitions.
overlap = development_hashes & evaluation_hashes

assert len(development_batches) == DEVELOPMENT_COUNT
assert len(evaluation_batch_indices) == EVALUATION_COUNT
assert not overlap

assert (
    max(partition_times["development"])
    < min(partition_times["evaluation"])
)

print("=== DATA PROVENANCE ===")
print("Source commit:", SOURCE_COMMIT)
print("Verified original files:", len(manifest))

print("\n=== DEVELOPMENT PARTITION ===")
print("Batches:", len(development_batches))
print("Unique measurements:", len(development_hashes))
print("First timestamp:", min(partition_times["development"]))
print("Last timestamp:", max(partition_times["development"]))

print("\n=== EVALUATION PARTITION ===")
print("Batches:", len(evaluation_batch_indices))
print("Unique measurements:", len(evaluation_hashes))
print("First timestamp:", min(partition_times["evaluation"]))
print("Last timestamp:", max(partition_times["evaluation"]))

print("\n=== PARTITION VALIDATION ===")
print("Overlapping records:", len(overlap))
print("Chronological separation: PASS")
print("Total referenced measurements:",
      len(development_hashes | evaluation_hashes))

# Retain development payloads only.
# Evaluation data remain available in the original source files.
del records_by_file, proofs, records, proof, timestamps, hashes

print("\nVALIDATION: PASS")

=== DATA PROVENANCE ===
Source commit: ec34ea5dc21642744d39eb436b787ac092505d0f
Verified original files: 12

=== DEVELOPMENT PARTITION ===
Batches: 65
Unique measurements: 3874
First timestamp: 2025-06-12 18:32:12.452010
Last timestamp: 2025-06-15 11:05:12.972425

=== EVALUATION PARTITION ===
Batches: 65
Unique measurements: 4211
First timestamp: 2025-06-15 11:06:12.972963
Last timestamp: 2025-06-18 09:16:13.539235

=== PARTITION VALIDATION ===
Overlapping records: 0
Chronological separation: PASS
Total referenced measurements: 8085

VALIDATION: PASS


In [2]:
# ============================================================
# CELL 2 — Canonical evidence representation
# ============================================================
# Construct common evidence for all verification architectures.
# Preserve original measurement strings and record order.
#
# Methodological reference:
# RFC 8785, Sections 3.1 and 3.2.
# https://doi.org/10.17487/RFC8785

from copy import deepcopy
from datetime import timezone

SCHEMA_VERSION = "iot-evidence-benchmark-v1"


def validate_json_types(value):
    """Restrict evidence to deterministic JSON-compatible types."""

    if type(value) is dict:

        for key, item in value.items():

            if type(key) is not str:
                raise TypeError("JSON keys must be strings.")

            validate_json_types(item)

    elif type(value) is list:

        for item in value:
            validate_json_types(item)

    elif type(value) not in (str, int, bool, type(None)):

        raise TypeError(
            f"Unsupported evidence type: {type(value).__name__}"
        )


def canonical_bytes(evidence):
    """Serialize evidence using the benchmark's fixed JSON rules."""

    validate_json_types(evidence)

    return json.dumps(
        evidence,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def evidence_commitment(evidence):
    """Calculate the SHA-256 commitment of canonical evidence."""

    return hashlib.sha256(
        canonical_bytes(evidence)
    ).hexdigest()


# Construct development evidence only.
development_evidence = []

previous_commitment = None

for batch in development_batches:

    records = deepcopy(batch["records"])

    source_ids = {
        record["device_id"] for record in records
    }

    assert len(source_ids) == 1

    # Synthetic control timestamp:
    # use the final measurement time as the application time.
    # This is not the original historical event timestamp.
    application_time = datetime.fromisoformat(
        records[-1]["timestamp"]
    ).replace(tzinfo=timezone.utc)

    evidence = {
        "schema": SCHEMA_VERSION,
        "sequence": batch["batch_index"],
        "source_id": next(iter(source_ids)),
        "previous_commitment": previous_commitment,
        "claims": {
            "application_timestamp_utc": (
                application_time.isoformat()
            ),
            "metadata_origin": "synthetic_control",
        },
        "records": records,
    }

    commitment = evidence_commitment(evidence)

    development_evidence.append({
        "evidence": evidence,
        "commitment": commitment,
    })

    previous_commitment = commitment


# ------------------------------------------------------------
# Independent implementation checks
# ------------------------------------------------------------

# Test the specified serialization against a fixed example.
assert canonical_bytes({"b": 2, "a": 1}) == b'{"a":1,"b":2}'

# Confirm that dictionary insertion order does not change bytes.
assert canonical_bytes({"a": 1, "b": 2}) == (
    canonical_bytes({"b": 2, "a": 1})
)

# Confirm that unsupported floating-point values are rejected.
try:
    canonical_bytes({"measurement": 0.1})
except TypeError:
    pass
else:
    raise AssertionError("Floating-point restriction failed.")

# Verify that every original record remains unchanged.
assert all(
    item["evidence"]["records"] == batch["records"]
    for item, batch in zip(
        development_evidence,
        development_batches,
        strict=True,
    )
)

# Verify all newly generated commitment links.
expected_previous = None

for item in development_evidence:

    evidence = item["evidence"]

    assert evidence["previous_commitment"] == expected_previous

    assert evidence_commitment(evidence) == item["commitment"]

    expected_previous = item["commitment"]

# Confirm that modifying evidence changes its commitment.
original = development_evidence[0]["evidence"]
modified = deepcopy(original)

modified["claims"]["application_timestamp_utc"] += "_modified"

assert evidence_commitment(original) != (
    evidence_commitment(modified)
)

# Confirm that record order affects the commitment.
reordered = deepcopy(original)
reordered["records"].reverse()

assert evidence_commitment(original) != (
    evidence_commitment(reordered)
)

commitments = {
    item["commitment"] for item in development_evidence
}

assert len(commitments) == len(development_evidence)

# Report results.
print("=== CANONICAL EVIDENCE ===")
print("Schema:", SCHEMA_VERSION)
print("Development evidence objects:", len(development_evidence))
print("Unique commitments:", len(commitments))

print("Measurement records:", sum(
    len(item["evidence"]["records"])
    for item in development_evidence
))

print("\n=== IMPLEMENTATION TESTS ===")
print("Reference serialization: PASS")
print("Dictionary order invariance: PASS")
print("Floating-point restriction: PASS")
print("Original record preservation: PASS")
print("Commitment chain consistency: PASS")
print("Metadata modification detection: PASS")
print("Record order sensitivity: PASS")

print("\n=== COMMITMENT BOUNDARIES ===")
print("First commitment:", development_evidence[0]["commitment"])
print("Last commitment:", development_evidence[-1]["commitment"])

print("\nVALIDATION: PASS")

=== CANONICAL EVIDENCE ===
Schema: iot-evidence-benchmark-v1
Development evidence objects: 65
Unique commitments: 65
Measurement records: 3874

=== IMPLEMENTATION TESTS ===
Reference serialization: PASS
Dictionary order invariance: PASS
Floating-point restriction: PASS
Original record preservation: PASS
Commitment chain consistency: PASS
Metadata modification detection: PASS
Record order sensitivity: PASS

=== COMMITMENT BOUNDARIES ===
First commitment: 33d5d385357f2975ba23e986c5d40fa41ac0b009bc203d210033e224132e42f5
Last commitment: 6b3496c7237cdc742fd1bc4f4b6959b498f675710d1b478b5a86fbabf6a16f85

VALIDATION: PASS


In [3]:
# ============================================================
# CELL 3 — Common digital signature mechanism
# ============================================================
# Implement Ed25519 signing, verification and authorization.
#
# Reference: RFC 8032, Sections 5.1 and 7.1.
# https://doi.org/10.17487/RFC8032
#
# WARNING:
# Deterministic keys are public experimental fixtures.
# They must never be used in production systems.

from Crypto.Signature import eddsa

SIGNATURE_DOMAIN = b"iot-evidence-benchmark-v1:source-signature\x00"


def experimental_key(role):
    """Generate a reproducible experimental Ed25519 key."""

    seed = hashlib.sha256(
        f"{SCHEMA_VERSION}:experimental-key:{role}".encode("utf-8")
    ).digest()

    return eddsa.import_private_key(seed)


def public_key_bytes(private_key):
    """Return the raw Ed25519 public key."""

    return private_key.public_key().export_key(format="raw")


def sign_evidence(evidence, private_key):
    """Sign the canonical evidence representation."""

    message = SIGNATURE_DOMAIN + canonical_bytes(evidence)

    return eddsa.new(
        private_key, "rfc8032"
    ).sign(message)


def verify_signature(evidence, signature, public_key):
    """Verify cryptographic authenticity independently of authorization."""

    if len(public_key) != 32 or len(signature) != 64:
        return False

    try:
        key = eddsa.import_public_key(public_key)

        message = SIGNATURE_DOMAIN + canonical_bytes(evidence)

        eddsa.new(key, "rfc8032").verify(
            message, signature
        )

        return True

    except (ValueError, TypeError):
        return False


# ------------------------------------------------------------
# Independent RFC 8032 reference test
# ------------------------------------------------------------

reference_seed = bytes.fromhex(
    "9d61b19deffd5a60ba844af492ec2cc4"
    "4449c5697b326919703bac031cae7f60"
)

reference_public_key = bytes.fromhex(
    "d75a980182b10ab7d54bfed3c964073a"
    "0ee172f3daa62325af021a68f707511a"
)

reference_signature = bytes.fromhex(
    "e5564300c360ac729086e2cc806e828a"
    "84877f1eb8e5d974d873e06522490155"
    "5fb8821590a33bacc61e39701cf9b46b"
    "d25bf5f0595bbe24655141438e7a100b"
)

reference_key = eddsa.import_private_key(reference_seed)

assert public_key_bytes(reference_key) == reference_public_key

assert eddsa.new(
    reference_key, "rfc8032"
).sign(b"") == reference_signature

eddsa.new(
    eddsa.import_public_key(reference_public_key),
    "rfc8032"
).verify(b"", reference_signature)


# ------------------------------------------------------------
# Experimental identities and authorization registry
# ------------------------------------------------------------

source_ids = {
    item["evidence"]["source_id"]
    for item in development_evidence
}

source_private_keys = {
    source_id: experimental_key(f"source:{source_id}")
    for source_id in sorted(source_ids)
}

# The registry is configured by the experiment,
# not supplied by the evidence issuer.
authorized_sources = {
    source_id: public_key_bytes(key)
    for source_id, key in source_private_keys.items()
}

unauthorized_key = experimental_key("unauthorized-source")


def is_authorized(evidence, public_key):
    """Check the configured source authorization policy."""

    return authorized_sources.get(
        evidence["source_id"]
    ) == public_key


# ------------------------------------------------------------
# Sign and verify all development evidence
# ------------------------------------------------------------

signed_development_evidence = []

for item in development_evidence:

    evidence = item["evidence"]
    source_id = evidence["source_id"]

    private_key = source_private_keys[source_id]
    public_key = authorized_sources[source_id]

    signature = sign_evidence(evidence, private_key)

    assert verify_signature(
        evidence, signature, public_key
    )

    assert is_authorized(evidence, public_key)

    signed_development_evidence.append({
        "evidence": deepcopy(evidence),
        "commitment": item["commitment"],
        "public_key": public_key,
        "signature": signature,
    })


# ------------------------------------------------------------
# Controlled positive and negative tests
# ------------------------------------------------------------

sample = signed_development_evidence[0]

original = sample["evidence"]
signature = sample["signature"]
public_key = sample["public_key"]

# Test 1: Valid signature.
assert verify_signature(original, signature, public_key)

# Test 2: Modified evidence.
modified = deepcopy(original)
modified["claims"]["application_timestamp_utc"] += "_modified"

assert not verify_signature(modified, signature, public_key)

# Test 3: Modified signature.
modified_signature = bytes([signature[0] ^ 1]) + signature[1:]

assert not verify_signature(
    original, modified_signature, public_key
)

# Test 4: Incorrect verification key.
other_public_key = public_key_bytes(unauthorized_key)

assert not verify_signature(
    original, signature, other_public_key
)

# Test 5: Valid signature from an unauthorized issuer.
other_signature = sign_evidence(original, unauthorized_key)

assert verify_signature(
    original, other_signature, other_public_key
)

assert not is_authorized(original, other_public_key)

# Test 6: Malformed signature.
assert not verify_signature(
    original, signature[:-1], public_key
)

# Test 7: Deterministic signing.
assert sign_evidence(
    original,
    source_private_keys[original["source_id"]]
) == signature


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("=== DIGITAL SIGNATURE CONFIGURATION ===")
print("Algorithm: Ed25519")
print("Authorized sources:", len(authorized_sources))
print("Public key length:", len(public_key), "bytes")
print("Signature length:", len(signature), "bytes")

print("\n=== REFERENCE VALIDATION ===")
print("RFC 8032 public key reproduction: PASS")
print("RFC 8032 signature reproduction: PASS")
print("RFC 8032 signature verification: PASS")

print("\n=== DEVELOPMENT EVIDENCE ===")
print("Signed batches:", len(signed_development_evidence))
print("Valid signatures:", len(signed_development_evidence))
print("Authorized signatures:", len(signed_development_evidence))

print("\n=== CONTROLLED TESTS ===")
print("Valid signature: PASS")
print("Modified evidence detection: PASS")
print("Modified signature detection: PASS")
print("Incorrect key detection: PASS")
print("Unauthorized issuer detection: PASS")
print("Malformed signature detection: PASS")
print("Deterministic signing: PASS")

print("\nVALIDATION: PASS")

=== DIGITAL SIGNATURE CONFIGURATION ===
Algorithm: Ed25519
Authorized sources: 1
Public key length: 32 bytes
Signature length: 64 bytes

=== REFERENCE VALIDATION ===
RFC 8032 public key reproduction: PASS
RFC 8032 signature reproduction: PASS
RFC 8032 signature verification: PASS

=== DEVELOPMENT EVIDENCE ===
Signed batches: 65
Valid signatures: 65
Authorized signatures: 65

=== CONTROLLED TESTS ===
Valid signature: PASS
Modified evidence detection: PASS
Modified signature detection: PASS
Incorrect key detection: PASS
Unauthorized issuer detection: PASS
Malformed signature detection: PASS
Deterministic signing: PASS

VALIDATION: PASS


In [4]:
# ============================================================
# CELL 4 — Transparency log and Merkle inclusion proofs
# ============================================================
# Implement the Merkle construction and inclusion verification
# specified in RFC 9162, Sections 2.1.1 and 2.1.3.
#
# Reference: https://doi.org/10.17487/RFC9162
#
# This is a research transparency log, not a complete
# Certificate Transparency protocol implementation.


def log_leaf_hash(entry):
    """Hash a log entry with leaf-domain separation."""

    return hashlib.sha256(b"\x00" + entry).digest()


def log_node_hash(left, right):
    """Hash two child nodes with node-domain separation."""

    return hashlib.sha256(
        b"\x01" + left + right
    ).digest()


def log_split(size):
    """Return the largest power of two smaller than size."""

    return 1 << ((size - 1).bit_length() - 1)


def log_tree_root(entries):
    """Calculate the RFC 9162 Merkle tree root."""

    size = len(entries)

    if size == 0:
        return hashlib.sha256(b"").digest()

    if size == 1:
        return log_leaf_hash(entries[0])

    k = log_split(size)

    return log_node_hash(
        log_tree_root(entries[:k]),
        log_tree_root(entries[k:])
    )


def log_inclusion_proof(entries, index):
    """Generate a Merkle inclusion path for one log entry."""

    size = len(entries)

    if not 0 <= index < size:
        raise ValueError("Invalid leaf index.")

    if size == 1:
        return []

    k = log_split(size)

    if index < k:
        return log_inclusion_proof(
            entries[:k], index
        ) + [log_tree_root(entries[k:])]

    return log_inclusion_proof(
        entries[k:], index - k
    ) + [log_tree_root(entries[:k])]


def verify_log_inclusion(entry, index, size, proof, root):
    """Verify inclusion using RFC 9162 Section 2.1.3.2."""

    if (
        not 0 <= index < size
        or len(root) != 32
        or any(len(node) != 32 for node in proof)
    ):
        return False

    fn = index
    sn = size - 1

    result = log_leaf_hash(entry)

    for node in proof:

        if sn == 0:
            return False

        if fn & 1 or fn == sn:

            result = log_node_hash(node, result)

            while fn and not (fn & 1):
                fn >>= 1
                sn >>= 1

        else:

            result = log_node_hash(result, node)

        fn >>= 1
        sn >>= 1

    return sn == 0 and result == root


# ------------------------------------------------------------
# Reference checks
# ------------------------------------------------------------

# Known SHA-256 digest of an empty byte string.
assert log_tree_root([]).hex() == (
    "e3b0c44298fc1c149afbf4c8996fb924"
    "27ae41e4649b934ca495991b7852b855"
)

# Single-leaf boundary condition.
assert log_inclusion_proof([b"a"], 0) == []

# Reproduce the seven-leaf tree structure illustrated
# in RFC 9162, Section 2.1.5.
example_entries = [
    f"d{i}".encode() for i in range(7)
]

# The seventh leaf requires two sibling subtree hashes.
assert log_inclusion_proof(example_entries, 6) == [
    log_tree_root(example_entries[4:6]),
    log_tree_root(example_entries[:4]),
]


# ------------------------------------------------------------
# Construct the development transparency log
# ------------------------------------------------------------

log_entries = [
    bytes.fromhex(item["commitment"])
    for item in signed_development_evidence
]

assert all(len(entry) == 32 for entry in log_entries)

assert len(set(log_entries)) == len(log_entries)

# Store the root at each chronological tree size.
log_roots = {
    size: log_tree_root(log_entries[:size])
    for size in range(len(log_entries) + 1)
}


# ------------------------------------------------------------
# Exhaustive development inclusion tests
# ------------------------------------------------------------

valid_tests = 0
negative_tests = 0

for size in range(1, len(log_entries) + 1):

    prefix = log_entries[:size]
    root = log_roots[size]

    for index, entry in enumerate(prefix):

        proof = log_inclusion_proof(prefix, index)

        # Positive test: correct inclusion proof.
        assert verify_log_inclusion(
            entry, index, size, proof, root
        )

        valid_tests += 1

        # Negative test: altered entry.
        assert not verify_log_inclusion(
            b"\xff" + entry, index, size, proof, root
        )

        # Negative test: excessive proof length.
        assert not verify_log_inclusion(
            entry, index, size,
            proof + [b"\x00" * 32], root
        )

        # Negative test: invalid leaf index.
        assert not verify_log_inclusion(
            entry, size, size, proof, root
        )

        negative_tests += 3

        # Negative test: incomplete proof.
        if proof:

            assert not verify_log_inclusion(
                entry, index, size, proof[:-1], root
            )

            negative_tests += 1


# ------------------------------------------------------------
# Report results
# ------------------------------------------------------------

print("=== TRANSPARENCY LOG CONFIGURATION ===")
print("Hash algorithm: SHA-256")
print("Log entries:", len(log_entries))
print("Unique log entries:", len(set(log_entries)))
print("Tree checkpoints:", len(log_roots))
print("Leaf and node domain separation: ENABLED")

print("\n=== REFERENCE VALIDATION ===")
print("Empty tree reference: PASS")
print("Single-leaf boundary: PASS")
print("RFC seven-leaf example: PASS")

print("\n=== INCLUSION VERIFICATION ===")
print("Positive tests passed:", valid_tests)
print("Negative tests passed:", negative_tests)
print("Total inclusion tests:", valid_tests + negative_tests)

print("\n=== TREE BOUNDARIES ===")
print("Empty root:", log_roots[0].hex())
print("First root:", log_roots[1].hex())
print("Final root:", log_roots[len(log_entries)].hex())

print("\nVALIDATION: PASS")

=== TRANSPARENCY LOG CONFIGURATION ===
Hash algorithm: SHA-256
Log entries: 65
Unique log entries: 65
Tree checkpoints: 66
Leaf and node domain separation: ENABLED

=== REFERENCE VALIDATION ===
Empty tree reference: PASS
Single-leaf boundary: PASS
RFC seven-leaf example: PASS

=== INCLUSION VERIFICATION ===
Positive tests passed: 2145
Negative tests passed: 8579
Total inclusion tests: 10724

=== TREE BOUNDARIES ===
Empty root: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
First root: d2fcb2dbb06f5f31d6ade7381e22f644ec6b1925aa7e0e27a737f641e027f676
Final root: 5c07462d9a839164c9831fb2aa784dd1adace2a0d1ea089f8b8903eee6e07928

VALIDATION: PASS


In [5]:
# ============================================================
# CELL 5 — Append-only Merkle consistency proofs
# ============================================================
# Implement consistency proof generation and verification.
#
# Reference: RFC 9162, Sections 2.1.4.1, 2.1.4.2 and 2.1.5.
# https://doi.org/10.17487/RFC9162
#
# Empty and equal-size trees are handled explicitly as
# benchmark boundary conditions.


def consistency_subproof(entries, old_size, known_root):
    """Recursively construct a Merkle consistency path."""

    if old_size == len(entries):
        return [] if known_root else [log_tree_root(entries)]

    k = log_split(len(entries))

    if old_size <= k:
        return consistency_subproof(
            entries[:k], old_size, known_root
        ) + [log_tree_root(entries[k:])]

    return consistency_subproof(
        entries[k:], old_size - k, False
    ) + [log_tree_root(entries[:k])]


def log_consistency_proof(entries, old_size):
    """Generate a proof of append-only extension."""

    if not 0 <= old_size <= len(entries):
        raise ValueError("Invalid historical tree size.")

    if old_size == 0:
        return []

    return consistency_subproof(entries, old_size, True)


def verify_log_consistency(
    old_size, new_size, old_root, new_root, proof
):
    """Verify consistency between two Merkle checkpoints."""

    if not 0 <= old_size <= new_size:
        return False

    if any(
        type(node) is not bytes or len(node) != 32
        for node in [old_root, new_root, *proof]
    ):
        return False

    # Empty tree: consistency is trivial, but the newer
    # checkpoint still requires independent authentication.
    if old_size == 0:
        return (
            old_root == log_tree_root([])
            and not proof
            and (new_size != 0 or new_root == old_root)
        )

    # Equal-sized trees must have identical roots.
    if old_size == new_size:
        return old_root == new_root and not proof

    if not proof:
        return False

    # RFC 9162: prepend the known historical root
    # when its tree size is an exact power of two.
    if old_size & (old_size - 1) == 0:
        path = [old_root, *proof]
    else:
        path = list(proof)

    first = old_size - 1
    second = new_size - 1

    while first & 1:
        first >>= 1
        second >>= 1

    reconstructed_old = path[0]
    reconstructed_new = path[0]

    for node in path[1:]:

        if second == 0:
            return False

        if first & 1 or first == second:

            reconstructed_old = log_node_hash(
                node, reconstructed_old
            )

            reconstructed_new = log_node_hash(
                node, reconstructed_new
            )

            while first and not (first & 1):
                first >>= 1
                second >>= 1

        else:

            reconstructed_new = log_node_hash(
                reconstructed_new, node
            )

        first >>= 1
        second >>= 1

    return (
        second == 0
        and reconstructed_old == old_root
        and reconstructed_new == new_root
    )


def flip_digest_bit(digest):
    """Introduce a controlled one-bit modification."""

    return bytes([digest[0] ^ 1]) + digest[1:]


# ------------------------------------------------------------
# RFC 9162 reference examples
# ------------------------------------------------------------

example = [f"d{i}".encode() for i in range(7)]

assert log_consistency_proof(example, 3) == [
    log_leaf_hash(example[2]),
    log_leaf_hash(example[3]),
    log_tree_root(example[:2]),
    log_tree_root(example[4:]),
]

assert log_consistency_proof(example, 4) == [
    log_tree_root(example[4:])
]

assert log_consistency_proof(example, 6) == [
    log_tree_root(example[4:6]),
    log_leaf_hash(example[6]),
    log_tree_root(example[:4]),
]


# ------------------------------------------------------------
# Boundary condition tests
# ------------------------------------------------------------

empty_root = log_roots[0]
first_root = log_roots[1]

assert verify_log_consistency(
    0, 0, empty_root, empty_root, []
)

assert verify_log_consistency(
    0, 1, empty_root, first_root, []
)

assert verify_log_consistency(
    1, 1, first_root, first_root, []
)

assert not verify_log_consistency(
    1, 1, first_root, flip_digest_bit(first_root), []
)

assert not verify_log_consistency(
    1, 0, first_root, empty_root, []
)


# ------------------------------------------------------------
# Exhaustive development consistency tests
# ------------------------------------------------------------

positive_consistency_tests = 0
negative_consistency_tests = 0

for new_size in range(1, len(log_entries) + 1):

    entries = log_entries[:new_size]
    new_root = log_roots[new_size]

    for old_size in range(1, new_size):

        old_root = log_roots[old_size]

        proof = log_consistency_proof(entries, old_size)

        # Positive: valid append-only extension.
        assert verify_log_consistency(
            old_size, new_size, old_root, new_root, proof
        )

        positive_consistency_tests += 1

        # Negative 1: modified historical root.
        assert not verify_log_consistency(
            old_size, new_size,
            flip_digest_bit(old_root), new_root, proof
        )

        # Negative 2: modified newer root.
        assert not verify_log_consistency(
            old_size, new_size,
            old_root, flip_digest_bit(new_root), proof
        )

        # Negative 3: incomplete proof.
        assert not verify_log_consistency(
            old_size, new_size,
            old_root, new_root, proof[:-1]
        )

        # Negative 4: excessive proof length.
        assert not verify_log_consistency(
            old_size, new_size,
            old_root, new_root, proof + [bytes(32)]
        )

        # Negative 5: modified proof node.
        modified_proof = [
            flip_digest_bit(proof[0]), *proof[1:]
        ]

        assert not verify_log_consistency(
            old_size, new_size,
            old_root, new_root, modified_proof
        )

        # Negative 6: internally valid but conflicting history.
        conflicting_entries = [
            flip_digest_bit(entries[0]), *entries[1:]
        ]

        conflicting_root = log_tree_root(conflicting_entries)

        conflicting_proof = log_consistency_proof(
            conflicting_entries, old_size
        )

        assert not verify_log_consistency(
            old_size, new_size,
            old_root, conflicting_root, conflicting_proof
        )

        negative_consistency_tests += 6


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("=== CONSISTENCY PROOF VALIDATION ===")
print("RFC reference examples: 3/3 PASS")
print("Boundary condition tests: 5/5 PASS")

print("\n=== DEVELOPMENT CONSISTENCY TESTS ===")
print("Log entries:", len(log_entries))
print("Positive tests:", positive_consistency_tests)
print("Negative tests:", negative_consistency_tests)

print("Total consistency tests:",
      positive_consistency_tests + negative_consistency_tests)

print("\n=== SECURITY PROPERTIES ===")
print("Valid append-only extensions: PASS")
print("Modified checkpoint rejection: PASS")
print("Incomplete proof rejection: PASS")
print("Excessive proof rejection: PASS")
print("Modified proof rejection: PASS")
print("Conflicting history rejection: PASS")

print("\nVALIDATION: PASS")

=== CONSISTENCY PROOF VALIDATION ===
RFC reference examples: 3/3 PASS
Boundary condition tests: 5/5 PASS

=== DEVELOPMENT CONSISTENCY TESTS ===
Log entries: 65
Positive tests: 2080
Negative tests: 12480
Total consistency tests: 14560

=== SECURITY PROPERTIES ===
Valid append-only extensions: PASS
Modified checkpoint rejection: PASS
Incomplete proof rejection: PASS
Excessive proof rejection: PASS
Modified proof rejection: PASS
Conflicting history rejection: PASS

VALIDATION: PASS


In [6]:
# ============================================================
# CELL 6 — Stateful witnesses and quorum verification
# ============================================================
# Implement authenticated checkpoints, stateful witnesses
# and the approved 3-of-4 witness policy.
#
# References:
# C2SP Transparency Log Witness Protocol v1.1.0
# https://c2sp.org/tlog-witness@v1.1.0
#
# RFC 9162, Section 2.1.4
# https://doi.org/10.17487/RFC9162
#
# This is a controlled research implementation.
# It is not a production witness service.

from itertools import combinations
import re

# Approved configuration: EXPERIMENTAL_DESIGN.md, Section 4.
WITNESS_COUNT = 4
WITNESS_QUORUM = 3
MAX_COMPROMISED_WITNESSES = 1

LOG_ID = "iot-evidence-transparency-log-v1"

OPERATOR_DOMAIN = b"iot-benchmark:log-checkpoint:v1\x00"
WITNESS_DOMAIN = b"iot-benchmark:witness-checkpoint:v1\x00"

operator_key = experimental_key("log-operator")
operator_public_key = public_key_bytes(operator_key)

witness_keys = {
    f"W{i}": experimental_key(f"witness:{i}")
    for i in range(1, WITNESS_COUNT + 1)
}

witness_public_keys = {
    name: public_key_bytes(key)
    for name, key in witness_keys.items()
}

assert len(set(witness_public_keys.values())) == WITNESS_COUNT


def make_checkpoint(entries):
    """Construct a checkpoint for the configured log."""

    return {
        "log_id": LOG_ID,
        "size": len(entries),
        "root": log_tree_root(entries).hex(),
    }


def valid_checkpoint(checkpoint):
    """Validate the checkpoint representation and log identity."""

    return (
        type(checkpoint) is dict
        and set(checkpoint) == {"log_id", "size", "root"}
        and checkpoint["log_id"] == LOG_ID
        and type(checkpoint["size"]) is int
        and checkpoint["size"] >= 0
        and type(checkpoint["root"]) is str
        and re.fullmatch(r"[0-9a-f]{64}", checkpoint["root"])
        is not None
        and (
            checkpoint["size"] != 0
            or checkpoint["root"] == log_tree_root([]).hex()
        )
    )


def sign_checkpoint(checkpoint, key, domain):
    """Sign a checkpoint using a role-specific signature domain."""

    message = domain + canonical_bytes(checkpoint)

    return eddsa.new(key, "rfc8032").sign(message)


def verify_checkpoint_signature(
    checkpoint, signature, public_key, domain
):
    """Verify checkpoint structure and signature."""

    if not valid_checkpoint(checkpoint):
        return False

    try:
        key = eddsa.import_public_key(public_key)

        eddsa.new(key, "rfc8032").verify(
            domain + canonical_bytes(checkpoint),
            signature,
        )

        return True

    except (ValueError, TypeError):
        return False


class Witness:
    """A stateful witness with independently retained history."""

    def __init__(self, name):

        self.name = name
        self.checkpoint = make_checkpoint([])

    def cosign(self, checkpoint, operator_signature, proof):

        # Authenticate the log operator.
        if not verify_checkpoint_signature(
            checkpoint,
            operator_signature,
            operator_public_key,
            OPERATOR_DOMAIN,
        ):
            return None

        # Verify consistency against retained witness state.
        if not verify_log_consistency(
            self.checkpoint["size"],
            checkpoint["size"],
            bytes.fromhex(self.checkpoint["root"]),
            bytes.fromhex(checkpoint["root"]),
            proof,
        ):
            return None

        # Sign and retain the newly verified checkpoint.
        signature = sign_checkpoint(
            checkpoint,
            witness_keys[self.name],
            WITNESS_DOMAIN,
        )

        self.checkpoint = deepcopy(checkpoint)

        return self.name, signature


def verify_quorum(checkpoint, operator_signature, cosignatures):
    """Require signatures from three distinct authorized witnesses."""

    if not verify_checkpoint_signature(
        checkpoint,
        operator_signature,
        operator_public_key,
        OPERATOR_DOMAIN,
    ):
        return False

    valid_witnesses = set()

    for name, signature in cosignatures:

        if name not in witness_public_keys:
            continue

        if verify_checkpoint_signature(
            checkpoint,
            signature,
            witness_public_keys[name],
            WITNESS_DOMAIN,
        ):
            valid_witnesses.add(name)

    return len(valid_witnesses) >= WITNESS_QUORUM


# ------------------------------------------------------------
# Validate the approved quorum configuration
# ------------------------------------------------------------

possible_quorums = list(
    combinations(witness_keys, WITNESS_QUORUM)
)

minimum_intersection = min(
    len(set(first) & set(second))
    for first, second in combinations(possible_quorums, 2)
)

assert minimum_intersection > MAX_COMPROMISED_WITNESSES


# ------------------------------------------------------------
# Construct witnessed development checkpoints
# ------------------------------------------------------------

witnesses = {
    name: Witness(name)
    for name in witness_keys
}

witnessed_checkpoints = []

for size in range(1, len(log_entries) + 1):

    entries = log_entries[:size]
    checkpoint = make_checkpoint(entries)

    operator_signature = sign_checkpoint(
        checkpoint, operator_key, OPERATOR_DOMAIN
    )

    cosignatures = []

    for witness in witnesses.values():

        proof = log_consistency_proof(
            entries, witness.checkpoint["size"]
        )

        result = witness.cosign(
            checkpoint, operator_signature, proof
        )

        assert result is not None

        cosignatures.append(result)

    assert verify_quorum(
        checkpoint, operator_signature, cosignatures
    )

    witnessed_checkpoints.append({
        "checkpoint": checkpoint,
        "operator_signature": operator_signature,
        "cosignatures": cosignatures,
    })


# ------------------------------------------------------------
# Quorum and availability tests
# ------------------------------------------------------------

sample = witnessed_checkpoints[-1]

checkpoint = sample["checkpoint"]
operator_signature = sample["operator_signature"]
cosignatures = sample["cosignatures"]

# Test all possible three-witness configurations.
for subset in combinations(cosignatures, WITNESS_QUORUM):

    assert verify_quorum(
        checkpoint, operator_signature, subset
    )

# Two witnesses must be insufficient.
for subset in combinations(cosignatures, WITNESS_QUORUM - 1):

    assert not verify_quorum(
        checkpoint, operator_signature, subset
    )

# Repeating one witness signature cannot create a quorum.
assert not verify_quorum(
    checkpoint,
    operator_signature,
    [cosignatures[0]] * WITNESS_QUORUM,
)

# A valid signature under another role is not a witness vote.
assert not verify_quorum(
    checkpoint,
    operator_signature,
    [
        *cosignatures[:2],
        (cosignatures[2][0], operator_signature),
    ],
)

# A modified operator signature must be rejected.
assert not verify_quorum(
    checkpoint,
    flip_digest_bit(operator_signature),
    cosignatures,
)

# Witnesses reject an older checkpoint.
older = witnessed_checkpoints[-2]

assert witnesses["W1"].cosign(
    older["checkpoint"],
    older["operator_signature"],
    [],
) is None


# ------------------------------------------------------------
# Controlled conflicting-history test
# ------------------------------------------------------------

# Construct two individually valid but conflicting histories.
history_a = log_entries[:2]

history_b = [
    flip_digest_bit(history_a[0]),
    history_a[1],
]

checkpoint_a = make_checkpoint(history_a)
checkpoint_b = make_checkpoint(history_b)

assert checkpoint_a["root"] != checkpoint_b["root"]

signature_a = sign_checkpoint(
    checkpoint_a, operator_key, OPERATOR_DOMAIN
)

signature_b = sign_checkpoint(
    checkpoint_b, operator_key, OPERATOR_DOMAIN
)

# Reset witness state for an independent scenario.
fork_witnesses = {
    name: Witness(name)
    for name in witness_keys
}

# W1 is modelled as compromised and signs both histories.
compromised_vote_a = (
    "W1",
    sign_checkpoint(
        checkpoint_a, witness_keys["W1"], WITNESS_DOMAIN
    ),
)

compromised_vote_b = (
    "W1",
    sign_checkpoint(
        checkpoint_b, witness_keys["W1"], WITNESS_DOMAIN
    ),
)

# Two honest witnesses accept history A.
vote_a2 = fork_witnesses["W2"].cosign(
    checkpoint_a, signature_a, []
)

vote_a3 = fork_witnesses["W3"].cosign(
    checkpoint_a, signature_a, []
)

assert vote_a2 is not None
assert vote_a3 is not None

assert verify_quorum(
    checkpoint_a,
    signature_a,
    [compromised_vote_a, vote_a2, vote_a3],
)

# Previously committed honest witnesses reject history B.
for name in ("W2", "W3"):

    assert fork_witnesses[name].cosign(
        checkpoint_b, signature_b, []
    ) is None

# The fourth witness has not previously observed history A.
vote_b4 = fork_witnesses["W4"].cosign(
    checkpoint_b, signature_b, []
)

assert vote_b4 is not None

# The conflicting history obtains only two votes.
assert not verify_quorum(
    checkpoint_b,
    signature_b,
    [compromised_vote_b, vote_b4],
)


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("=== WITNESS CONFIGURATION ===")
print("Registered witnesses:", len(witnesses))
print("Required quorum:", WITNESS_QUORUM)
print("Maximum assumed compromised witnesses:",
      MAX_COMPROMISED_WITNESSES)
print("Possible quorums:", len(possible_quorums))
print("Minimum quorum intersection:", minimum_intersection)

print("\n=== DEVELOPMENT CHECKPOINTS ===")
print("Witnessed checkpoints:", len(witnessed_checkpoints))
print("Valid witness signatures:", sum(
    len(item["cosignatures"])
    for item in witnessed_checkpoints
))
print("Accepted checkpoint quorums:",
      len(witnessed_checkpoints))

print("\n=== CONTROLLED SECURITY TESTS ===")
print("Three-witness availability: PASS")
print("Two-witness rejection: PASS")
print("Duplicate witness rejection: PASS")
print("Signature domain separation: PASS")
print("Modified operator signature rejection: PASS")
print("Historical checkpoint rollback rejection: PASS")
print("Conflicting checkpoint rejection: PASS")
print("Single compromised witness fork test: PASS")

print("\nVALIDATION: PASS")

=== WITNESS CONFIGURATION ===
Registered witnesses: 4
Required quorum: 3
Maximum assumed compromised witnesses: 1
Possible quorums: 4
Minimum quorum intersection: 2

=== DEVELOPMENT CHECKPOINTS ===
Witnessed checkpoints: 65
Valid witness signatures: 260
Accepted checkpoint quorums: 65

=== CONTROLLED SECURITY TESTS ===
Three-witness availability: PASS
Two-witness rejection: PASS
Duplicate witness rejection: PASS
Signature domain separation: PASS
Modified operator signature rejection: PASS
Historical checkpoint rollback rejection: PASS
Conflicting checkpoint rejection: PASS
Single compromised witness fork test: PASS

VALIDATION: PASS


In [9]:
# ============================================================
# CELL 7 — Local Ethereum anchoring architecture
# ============================================================
# Compile, deploy and test an append-only commitment contract.
#
# References:
# https://pypi.org/project/eth-tester/0.14.0b1/
# https://docs.soliditylang.org/en/v0.8.36/
#
# Local EVM execution does not establish decentralized
# consensus security or public-network performance.

import subprocess
from statistics import median
from eth_tester import EthereumTester, PyEVMBackend
from eth_tester.exceptions import TransactionFailed
from eth_hash.auto import keccak
from eth_abi import encode, decode

COMPILER_DIR = ROOT / "tools" / "solidity"
SOLIDITY_VERSION = "0.8.36"

# Execution budget, not a measured transaction cost.
GAS_LIMIT = 300_000
ZERO_COMMITMENT = bytes(32)

CONTRACT_SOURCE = """
// SPDX-License-Identifier: MIT
pragma solidity 0.8.36;

contract EvidenceAnchor {

    address public immutable relay;

    uint256 public count;
    bytes32 public head;

    mapping(uint256 => bytes32) public commitments;

    event Anchored(
        uint256 indexed sequence,
        bytes32 commitment,
        bytes32 previousCommitment
    );

    constructor() {
        relay = msg.sender;
    }

    function append(
        uint256 sequence,
        bytes32 commitment,
        bytes32 previousCommitment
    ) external {

        require(msg.sender == relay, "unauthorized relay");
        require(sequence == count, "invalid sequence");
        require(previousCommitment == head, "invalid predecessor");
        require(commitment != bytes32(0), "empty commitment");

        commitments[sequence] = commitment;

        head = commitment;
        count += 1;

        emit Anchored(
            sequence,
            commitment,
            previousCommitment
        );
    }
}
"""


# ------------------------------------------------------------
# Compile using the preserved Solidity environment
# ------------------------------------------------------------

def compile_anchor_contract():

    compiler_version = subprocess.check_output(
        ["node", "-e", "console.log(require('solc').version())"],
        cwd=COMPILER_DIR,
        text=True,
    ).strip()

    assert compiler_version.startswith(SOLIDITY_VERSION + "+")

    compiler_input = {
        "language": "Solidity",
        "sources": {
            "EvidenceAnchor.sol": {
                "content": CONTRACT_SOURCE
            }
        },
        "settings": {
            "evmVersion": "cancun",
            "optimizer": {"enabled": False},
            "outputSelection": {
                "*": {
                    "*": ["abi", "evm.bytecode.object"]
                }
            },
        },
    }

    output = subprocess.check_output(
        [
            "node", "-e",
            """
            const fs = require('fs');
            const solc = require('solc');
            process.stdout.write(
                solc.compile(fs.readFileSync(0, 'utf8'))
            );
            """,
        ],
        cwd=COMPILER_DIR,
        input=json.dumps(compiler_input),
        text=True,
    )

    compiled = json.loads(output)

    errors = [
        item for item in compiled.get("errors", [])
        if item["severity"] == "error"
    ]

    assert not errors, errors

    contract = compiled["contracts"][
        "EvidenceAnchor.sol"
    ]["EvidenceAnchor"]

    return contract["evm"]["bytecode"]["object"]


anchor_bytecode = compile_anchor_contract()


# ------------------------------------------------------------
# Local Ethereum interface
# ------------------------------------------------------------

def contract_calldata(function, types=(), values=()):
    """Encode a Solidity function call."""

    selector = keccak(function.encode("utf-8"))[:4]

    arguments = encode(list(types), list(values))

    return "0x" + (selector + arguments).hex()


class EthereumAnchor:
    """Execute and query the commitment contract locally."""

    def __init__(self):

        self.tester = EthereumTester(
            backend=PyEVMBackend()
        )

        self.relay = self.tester.get_accounts()[0]
        self.other_account = self.tester.get_accounts()[1]

        deployment = {
            "from": self.relay,
            "data": "0x" + anchor_bytecode,
        }

        deployment["gas"] = self.tester.estimate_gas(
            deployment
        )

        tx_hash = self.tester.send_transaction(deployment)

        self.deployment_receipt = (
            self.tester.get_transaction_receipt(tx_hash)
        )

        assert self.deployment_receipt["status"] == 1

        self.address = (
            self.deployment_receipt["contract_address"]
        )

    def query(self, function, result_type, types=(), values=()):

        result = self.tester.call({
            "from": self.relay,
            "to": self.address,
            "data": contract_calldata(
                function, types, values
            ),
        })

        return decode(
            [result_type],
            bytes.fromhex(result[2:])
        )[0]

    def count(self):
        return self.query("count()", "uint256")

    def head(self):
        return self.query("head()", "bytes32")

    def commitment_at(self, sequence):

        if not 0 <= sequence < self.count():
            return None

        return self.query(
            "commitments(uint256)",
            "bytes32",
            ["uint256"],
            [sequence],
        )

    def append_transaction(
        self, sequence, commitment, previous, sender=None
    ):

        return {
            "from": sender or self.relay,
            "to": self.address,
            "data": contract_calldata(
                "append(uint256,bytes32,bytes32)",
                ["uint256", "bytes32", "bytes32"],
                [sequence, commitment, previous],
            ),
            "gas": GAS_LIMIT,
        }

    def append(
        self, sequence, commitment, previous, sender=None
    ):

        transaction = self.append_transaction(
            sequence, commitment, previous, sender
        )

        tx_hash = self.tester.send_transaction(transaction)

        return self.tester.get_transaction_receipt(tx_hash)


# ------------------------------------------------------------
# Deploy and validate initial state
# ------------------------------------------------------------

ethereum_anchor = EthereumAnchor()

assert ethereum_anchor.count() == 0
assert ethereum_anchor.head() == ZERO_COMMITMENT
assert ethereum_anchor.commitment_at(0) is None

assert ethereum_anchor.query(
    "relay()", "address"
).lower() == ethereum_anchor.relay.lower()


# ------------------------------------------------------------
# Anchor all development evidence
# ------------------------------------------------------------

ethereum_receipts = []

event_topic = "0x" + keccak(
    b"Anchored(uint256,bytes32,bytes32)"
).hex()

for item in signed_development_evidence:

    evidence = item["evidence"]

    # Apply the same source authentication used by A and B.
    assert verify_signature(
        evidence, item["signature"], item["public_key"]
    )

    assert is_authorized(evidence, item["public_key"])

    sequence = evidence["sequence"]

    commitment = bytes.fromhex(
        evidence_commitment(evidence)
    )

    assert commitment.hex() == item["commitment"]

    previous = (
        ZERO_COMMITMENT
        if evidence["previous_commitment"] is None
        else bytes.fromhex(evidence["previous_commitment"])
    )

    receipt = ethereum_anchor.append(
        sequence, commitment, previous
    )

    assert receipt["status"] == 1

    # Verify actual contract storage.
    assert ethereum_anchor.commitment_at(sequence) == commitment
    assert ethereum_anchor.head() == commitment
    assert ethereum_anchor.count() == sequence + 1

    # Independently inspect the emitted event.
    assert len(receipt["logs"]) == 1

    event = receipt["logs"][0]

    assert event["address"].lower() == (
        ethereum_anchor.address.lower()
    )

    assert len(event["topics"]) == 2
    assert event["topics"][0] == event_topic
    assert int(event["topics"][1], 16) == sequence

    event_commitment, event_previous = decode(
        ["bytes32", "bytes32"],
        bytes.fromhex(event["data"][2:])
    )

    assert event_commitment == commitment
    assert event_previous == previous

    ethereum_receipts.append(receipt)


# ------------------------------------------------------------
# Verify failed transactions and unchanged state
# ------------------------------------------------------------

def anchor_state(anchor):
    """Read the complete current commitment history."""

    return (
        anchor.count(),
        anchor.head(),
        tuple(
            anchor.commitment_at(i)
            for i in range(anchor.count())
        ),
    )


original_state = anchor_state(ethereum_anchor)

next_sequence = ethereum_anchor.count()
current_head = ethereum_anchor.head()

candidate_commitment = hashlib.sha256(
    b"development-negative-test"
).digest()

# Confirm that the gas budget is sufficient for a valid append.
valid_transaction = ethereum_anchor.append_transaction(
    next_sequence, candidate_commitment, current_head
)

assert ethereum_anchor.tester.estimate_gas(
    valid_transaction
) < GAS_LIMIT


def check_rejected_append(
    name, sequence, commitment, previous,
    expected_reason, sender=None
):

    transaction = ethereum_anchor.append_transaction(
        sequence, commitment, previous, sender
    )

    # Verify the intended failure reason before execution.
    try:
        ethereum_anchor.tester.call(transaction)

    except TransactionFailed as error:
        assert expected_reason in str(error), str(error)

    else:
        raise AssertionError("Invalid append did not revert.")

    # Submit an actual transaction and inspect its receipt.
    tx_hash = ethereum_anchor.tester.send_transaction(
        transaction
    )

    receipt = ethereum_anchor.tester.get_transaction_receipt(
        tx_hash
    )

    assert receipt["status"] == 0
    assert receipt["gas_used"] < GAS_LIMIT
    assert not receipt["logs"]

    # A failed append must not modify any stored commitment.
    assert anchor_state(ethereum_anchor) == original_state

    print(f"{name}: PASS")


print("=== CONTROLLED CONTRACT TESTS ===")

check_rejected_append(
    "Duplicate sequence rejection",
    0, log_entries[0], ZERO_COMMITMENT,
    "invalid sequence",
)

check_rejected_append(
    "Existing commitment overwrite rejection",
    0, candidate_commitment, ZERO_COMMITMENT,
    "invalid sequence",
)

check_rejected_append(
    "Skipped sequence rejection",
    next_sequence + 1, candidate_commitment, current_head,
    "invalid sequence",
)

check_rejected_append(
    "Incorrect predecessor rejection",
    next_sequence, candidate_commitment,
    flip_digest_bit(current_head),
    "invalid predecessor",
)

check_rejected_append(
    "Empty commitment rejection",
    next_sequence, ZERO_COMMITMENT, current_head,
    "empty commitment",
)

check_rejected_append(
    "Unauthorized relay rejection",
    next_sequence, candidate_commitment, current_head,
    "unauthorized relay",
    sender=ethereum_anchor.other_account,
)


# ------------------------------------------------------------
# Final comparison against the common commitment history
# ------------------------------------------------------------

assert ethereum_anchor.count() == len(log_entries)

assert [
    ethereum_anchor.commitment_at(i)
    for i in range(len(log_entries))
] == log_entries

assert ethereum_anchor.head() == log_entries[-1]

gas_values = [
    receipt["gas_used"] for receipt in ethereum_receipts
]

print("\n=== ETHEREUM ANCHORING ===")
print("Contract deployment: PASS")
print("Anchored commitments:", ethereum_anchor.count())
print("Successful transactions:", len(ethereum_receipts))
print("Matching stored commitments:", len(log_entries))
print("Verified anchor events:", len(ethereum_receipts))
print("Final commitment:", ethereum_anchor.head().hex())

print("\n=== EXECUTION GAS ===")
print("Deployment:", ethereum_anchor.deployment_receipt["gas_used"])
print("Minimum append:", min(gas_values))
print("Median append:", median(gas_values))
print("Maximum append:", max(gas_values))
print("Total append gas:", sum(gas_values))

print("\n=== FINAL VALIDATION ===")
print("Controlled rejection tests: 6/6 PASS")
print("Historical state preservation: PASS")
print("Common commitment equivalence: PASS")

print("\nVALIDATION: PASS")

=== CONTROLLED CONTRACT TESTS ===
Duplicate sequence rejection: PASS
Existing commitment overwrite rejection: PASS
Skipped sequence rejection: PASS
Incorrect predecessor rejection: PASS
Empty commitment rejection: PASS
Unauthorized relay rejection: PASS

=== ETHEREUM ANCHORING ===
Contract deployment: PASS
Anchored commitments: 65
Successful transactions: 65
Matching stored commitments: 65
Verified anchor events: 65
Final commitment: 6b3496c7237cdc742fd1bc4f4b6959b498f675710d1b478b5a86fbabf6a16f85

=== EXECUTION GAS ===
Deployment: 440947
Minimum append: 57530
Median append: 57554
Maximum append: 91346
Total append gas: 3774574

=== FINAL VALIDATION ===
Controlled rejection tests: 6/6 PASS
Historical state preservation: PASS
Common commitment equivalence: PASS

VALIDATION: PASS


In [10]:
# ============================================================
# TEMPORARY CELL — Evidence-validation requirements audit
# ============================================================
# Inspect the approved requirements and development data.
# Read-only operation. No evaluation data are accessed.

from pathlib import Path
import csv
import json

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "PROTOCOL.md").is_file()
)

# Inspect approved methodological documents.
for filename in (
    "REQUIREMENTS.md",
    "EXPERIMENTAL_DESIGN.md",
    "PROTOCOL_ADDENDUM.md",
    "PROTOCOL_AMENDMENT_01.md",
):
    path = root / filename

    assert path.is_file(), f"Missing document: {filename}"

    print(f"\n=== {filename} ===")
    print(path.read_text(encoding="utf-8"))

# Inspect one development measurement record.
raw = root / "data" / "raw"

proofs = json.loads(
    (raw / "Meta Data/proof_log.json").read_text()
)

first_proof = proofs[0]

csv_file = raw / "logs" / Path(
    first_proof["csv_path"]
).name

with csv_file.open(
    encoding="utf-8", newline=""
) as stream:

    rows = csv.DictReader(stream)

    first_record = next(
        row for index, row in enumerate(rows)
        if index == first_proof["row_start"]
    )

print("\n=== ORIGINAL MEASUREMENT REPRESENTATION ===")
print(json.dumps(first_record, indent=2))

print("\n=== ORIGINAL BATCH METADATA ===")
print(json.dumps(first_proof, indent=2))

print("\n=== AUDIT STATUS ===")
print("Approved documents inspected: 4")
print("Development sample inspected: 1")
print("Evaluation payloads accessed: 0")
print("Files modified: 0")

print("\nVALIDATION: PASS")


=== REQUIREMENTS.md ===
# Scientific and Technical Requirements

Status: Draft
Version: 0.1

## 1. Assurance Requirements

R01 - Data integrity
Detect changes to cryptographically committed records.

R02 - Source authentication
Verify the identity associated with a signing key.
Evaluate authorization separately from authentication.

R03 - Inclusion
Verify whether a record belongs to a committed batch.

R04 - History consistency
Evaluate whether successive commitments preserve
an append-only history.

R05 - Non-equivocation
Evaluate conflicting histories presented to
different relying parties under explicit trust assumptions.

R06 - Evidence completeness
Evaluate whether the required records and supporting
evidence are available.

R07 - Freshness
Evaluate evidence against the applicable time requirements.

R08 - Semantic validity
Evaluate data structure, measurement units and
documented consistency constraints.

R09 - Institutional validity
Evaluate applicable issuer authorization, cer